## qEEG(밴드파워 CSV 기반) — SIVD(VaD) vs MCI_vascular 분류, raw EEG 불필요 버전

앞의 두 노트북(`try1.ipynb`, `Vascular_mci_hc_vd.ipynb`)은 raw EEG를 직접 로딩해서 윈도우 단위로
`qeeg_bandpower_features()`를 매번 계산했지만, 이 노트북은 **이미 계산되어 있는 피험자 단위 밴드파워
CSV(`SIVD_vascMCI_QEEG.csv`)를 그대로 읽어서** 분류만 수행하는 훨씬 가벼운 버전이다. raw `.set` 파일이나
MNE가 전혀 필요 없어서 빠르게 재현/검증할 때 유용함.

- **데이터 병합**: `subject_list.csv`(라벨)와 `SIVD_vascMCI_QEEG.csv`(피험자별 qEEG 지표)를
  `sample_id` ↔ `Subject` 기준으로 inner join. `assert len(df) == len(sub) == len(qeeg)`로 두 CSV의
  피험자 수가 정확히 일치하는지 강제 검증(하나라도 안 맞으면 조용히 넘어가지 않고 바로 에러) — 다른
  노트북들에서 실패 목록을 모아 print만 하고 넘어가는 것과 달리 여기는 fail-fast 방식.
- **라벨**: SIVD(VaD)=1, MCI_vascular=0 — Cell 0/1과 동일한 규칙 유지.
- **피처(8개, 전부 "All_Channels" 접두사 = 19채널 평균/전체 기준 지표)**:
  - 상대 밴드파워 5개: Delta / Theta / Alpha / Beta / Gamma
  - 비율 지표 3개: DAR(Delta/Alpha Ratio), TAR(Theta/Alpha Ratio), DTABR((Delta+Theta)/(Alpha+Beta) Ratio)
    — 전부 임상 qEEG 문헌에서 "서파화(slowing)" 정도를 나타내는 데 쓰이는 표준 지표들.
- **CV**: `StratifiedKFold(n_splits=5)` × 5 repeat. 다른 노트북들이 윈도우 단위 데이터라
  `StratifiedGroupKFold`(subject leakage 방지)를 쓰는 것과 달리, 여기는 **피험자당 이미 1개의 피처
  행**이라 애초에 subject-level leakage 가능성이 없으므로 group 분리 없이 일반 `StratifiedKFold`로 충분.
- **전처리**: fold마다 `StandardScaler`를 train set에만 `fit`하고 train/test 둘 다 `transform`
  — test 정보가 스케일링에 섞여 들어가는 leakage 방지.
- **모델**: `LogisticRegression(max_iter=2000, class_weight='balanced')` — 다른 노트북의 qEEG
  파이프라인과 동일한 설정.
- **결과**: `AUC=0.662±0.093, acc=0.626±0.087, macroF1=0.621±0.088` (25-fold, 5×5)

> 📌 참고: 이 결과(AUC=0.662±0.093)는 이전에 raw EEG 기반 파이프라인의 서로 다른 소스(`내용정리.txt`의
> "결과 공유" 섹션)에서 확인했던 VD-VMCI 과제의 qEEG "all/broadband" 베이스라인 수치와 **완전히 일치**한다.
> 서로 다른 두 파이프라인(윈도우 단위 Welch PSD 계산 vs 사전 계산된 subject 단위 CSV)에서 독립적으로
> 같은 값이 나왔다는 건, 예전 초록에 잘못 들어갔던 "0.655"가 아니라 **0.662가 실제로 맞는 qEEG 수치**라는
> 점을 한 번 더 뒷받침하는 교차 검증 결과다.

⚠️ `DATA_ROOT`가 `'D:/Downloads/Vascular_Subset/'`로 되어 있어 `Vascular_mci_hc_vd.ipynb`의
`'C:/eeg_research/Vascular_Subset/'`와 다름 — 같은 저장소에 함께 올릴 거라면 경로 표기를 통일하거나,
"이 노트북은 다른 컴퓨터/시점에서 작성됨" 같은 짧은 주석을 남겨두는 게 좋음.

In [ ]:
"""
qEEG(밴드파워) 기반 SIVD(VaD) vs MCI_vascular 이진분류 — Logistic Regression
subject_list.csv + SIVD_vascMCI_QEEG.csv만 있으면 됨 (raw EEG 파일 불필요)
"""
import pandas as pd, numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

# ---------------- 경로 설정 ----------------
DATA_ROOT = 'C:/eeg_research/Vascular_Subset/'
SUBJECT_LIST_CSV = DATA_ROOT + 'subject_list.csv'
QEEG_CSV = DATA_ROOT + 'SIVD_vascMCI_QEEG.csv'

N_FOLDS = 5
N_REPEATS = 5

sub = pd.read_csv(SUBJECT_LIST_CSV)
qeeg = pd.read_csv(QEEG_CSV)

sub['folder'] = sub['subject_id'].astype(str).str.zfill(5)
sub['label'] = (sub['group'].str.lower() == 'sivd').astype(int)  # 1=SIVD(VaD), 0=mci_vascular

df = sub.merge(qeeg, left_on='sample_id', right_on='Subject', how='inner')
assert len(df) == len(sub) == len(qeeg), f'병합 후 개수 불일치: sub={len(sub)} qeeg={len(qeeg)} merged={len(df)}'
print(f'총 {len(df)}명 (VaD/SIVD={df["label"].sum()}, MCI_vascular={len(df)-df["label"].sum()})')

feat_cols = ['All_Channels_RelPow_Delta', 'All_Channels_RelPow_Theta', 'All_Channels_RelPow_Alpha',
             'All_Channels_RelPow_Beta', 'All_Channels_RelPow_Gamma',
             'All_Channels_Ratio_DAR', 'All_Channels_Ratio_TAR', 'All_Channels_Ratio_DTABR']
X = df[feat_cols].values.astype(np.float64)
y = df['label'].values
assert not np.isnan(X).any(), 'feature에 NaN 있음'

accs, aucs, f1s = [], [], []
for rep in range(N_REPEATS):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=rep)
    for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y)):
        scaler = StandardScaler().fit(X[tr_idx])
        Xtr, Xte = scaler.transform(X[tr_idx]), scaler.transform(X[te_idx])
        clf = LogisticRegression(max_iter=2000, class_weight='balanced')
        clf.fit(Xtr, y[tr_idx])
        proba = clf.predict_proba(Xte)[:, 1]
        pred = clf.predict(Xte)
        auc = roc_auc_score(y[te_idx], proba)
        acc = accuracy_score(y[te_idx], pred)
        f1 = f1_score(y[te_idx], pred, average='macro')
        accs.append(acc); aucs.append(auc); f1s.append(f1)
        print(f'[rep {rep+1}/{N_REPEATS}][fold {fold+1}/{N_FOLDS}] AUC={auc:.3f} acc={acc:.3f} f1={f1:.3f} '
              f'(train={len(tr_idx)} test={len(te_idx)})')

print(f'\n[qEEG-only] AUC={np.mean(aucs):.3f}±{np.std(aucs):.3f}  acc={np.mean(accs):.3f}±{np.std(accs):.3f}  '
      f'macroF1={np.mean(f1s):.3f}±{np.std(f1s):.3f}')

총 132명 (VaD/SIVD=71, MCI_vascular=61)
[rep 1/5][fold 1/5] AUC=0.656 acc=0.630 f1=0.629 (train=105 test=27)
[rep 1/5][fold 2/5] AUC=0.522 acc=0.630 f1=0.629 (train=105 test=27)
[rep 1/5][fold 3/5] AUC=0.726 acc=0.731 f1=0.727 (train=106 test=26)
[rep 1/5][fold 4/5] AUC=0.744 acc=0.654 f1=0.649 (train=106 test=26)
[rep 1/5][fold 5/5] AUC=0.643 acc=0.577 f1=0.561 (train=106 test=26)
[rep 2/5][fold 1/5] AUC=0.561 acc=0.481 f1=0.481 (train=105 test=27)
[rep 2/5][fold 2/5] AUC=0.775 acc=0.778 f1=0.777 (train=105 test=27)
[rep 2/5][fold 3/5] AUC=0.589 acc=0.500 f1=0.499 (train=106 test=26)
[rep 2/5][fold 4/5] AUC=0.863 acc=0.769 f1=0.768 (train=106 test=26)
[rep 2/5][fold 5/5] AUC=0.690 acc=0.692 f1=0.692 (train=106 test=26)
[rep 3/5][fold 1/5] AUC=0.611 acc=0.593 f1=0.583 (train=105 test=27)
[rep 3/5][fold 2/5] AUC=0.731 acc=0.704 f1=0.700 (train=105 test=27)
[rep 3/5][fold 3/5] AUC=0.744 acc=0.692 f1=0.692 (train=106 test=26)
[rep 3/5][fold 4/5] AUC=0.464 acc=0.423 f1=0.422 (train=106 test=